> ⚠️ **Sandbox notice** — This notebook runs against the FiGuard shared demo environment (`sb_live_demo` key, `https://figuard-sandbox-g1ha.onrender.com`). It is **not for production use**. Budgets, events, and tokens may be wiped at any time without notice to keep the sandbox available for everyone. No uptime SLA. [Self-host for production →](https://github.com/figuard/figuard-core#self-hosting)

> 💡 **Quick start:** Select **Runtime → Run all** (Ctrl+F9) to run everything at once, then read the output below.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys
!pip install --upgrade "figuard[langchain]>=0.3.0" langchain-openai -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + LangChain ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (identifies your budgets in the dashboard)")

In [ ]:
from langchain_openai import ChatOpenAI
from figuard.integrations.langchain import FiGuardCallbackHandler
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    intent_context="shopping assistant session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

session_token = budget.session_token

In [ ]:
# Wire FiGuard into the LangChain callback stack
# Every tool call is pre-flight authorized before it executes
import os

handler = FiGuardCallbackHandler(
    client=client,
    session_token=session_token,
    agent_id="shopping_agent",
)

# OPENAI_API_KEY must be set for the LLM to run
# os.environ["OPENAI_API_KEY"] = "your-key-here"

import os
_api_key = os.environ.get("OPENAI_API_KEY", "sk-demo-not-used")
llm = ChatOpenAI(api_key=_api_key, callbacks=[handler])
# Every tool call the LLM makes is now gated through FiGuard
# Denied calls raise FiGuardDeniedError before the tool executes

print("✓ LangChain + FiGuard wired up")
print("  Set OPENAI_API_KEY and invoke your chain to see live enforcement.")
print(f"\n→ See the spend tree: https://figuard-sandbox-g1ha.onrender.com/ui/budgets/{budget.id}")